# Lab 1 — Tokens (EP01)

ในคลิปทฤษฎีเราเห็นว่า LLM อ่านและเขียนเป็น **token** และทุกอย่างคิดเงินเป็น token
lab นี้เราจะเปิดดูของจริงกับ **Gemma 4 26B** ผ่าน OpenRouter

**สิ่งที่จะได้เรียน**
1. เรียก LLM ผ่าน API และอ่าน `usage` (จำนวน token ที่ใช้)
2. ดูว่าข้อความถูกตัดเป็น token อย่างไร — ภาษาไทยเทียบกับภาษาอังกฤษ
3. นับ token จริงของ Gemma รวมถึง token ที่ chat template เพิ่มให้
4. คำนวณค่าใช้จ่าย เช็ก context window และดูคำตอบที่ถูกตัดเพราะ `max_tokens`

ข้อความตัวอย่างใน lab นี้ (abstract) เป็น **ข้อมูลจำลองที่เขียนขึ้นเพื่อการสอน** ส่วนตัวเลขและคำตอบของ model ทั้งหมดมาจากการรันจริง

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.1–1.2, §18.2 — CC BY-SA 4.0

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **5 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep01-lab1-tokens.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## 0. เตรียมเครื่องก่อนเริ่ม

| สิ่งที่ต้องมี | รายละเอียด |
|---|---|
| Python 3.10 ขึ้นไป | หรือใช้ Google Colab ซึ่งมีให้แล้ว |
| OpenRouter API key | สมัครที่ openrouter.ai → Keys → Create Key แล้วเติมเครดิต (ทั้ง EP ใช้ไม่ถึง 1 บาท) |
| Packages | `openai`, `python-dotenv`, `tiktoken`, `requests` (ติดตั้งใน cell ถัดไป) |

**ใส่ API key**
- **Colab:** ไอคอน 🔑 Secrets → ชื่อ `OPENROUTER_API_KEY` → เปิด *Notebook access*
- **เครื่องตัวเอง:** สร้างไฟล์ `.env` ในโฟลเดอร์เดียวกับ notebook แล้วใส่บรรทัด `OPENROUTER_API_KEY=sk-or-...`

⚠️ อย่าใส่ key ลงในโค้ดโดยตรง และอย่าแชร์ไฟล์ `.env`

In [ ]:
%pip install -q "openai>=1.50" python-dotenv tiktoken requests

## 1. เรียก Gemma ผ่าน OpenRouter

**ทำไมใช้ `google/gemma-4-26b-a4b-it`** — instruction-tuned (ทำตามคำสั่งได้), MoE ที่ทำงานจริง ~4B ต่อ token จึงเร็วและถูก,
คืนความน่าจะเป็นของ token ได้ (ใช้ใน Lab 2) และรับรูปภาพได้ (ใช้ใน Lab 3)

**ไม่ฟรี** แต่ถูกมาก: ทั้ง EP01 ใช้ไม่ถึงหนึ่งบาท · รุ่น `:free` มี provider รายเดียวที่ไม่คืนความน่าจะเป็น
และถูกจำกัดจำนวนครั้ง (ทดสอบ 2026-11-08: ตอบ 429 "rate-limited" 1 ใน 3 ครั้ง และ error 404 ทุกครั้งที่ขอความน่าจะเป็น)

In [ ]:
import os, requests, tiktoken
from openai import OpenAI

In [ ]:
def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False}}

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "ในโลกของ LLM token คืออะไร ตอบในประโยคเดียว"}],
    extra_body=EXTRA,
)
print(resp.choices[0].message.content)
print(resp.usage.prompt_tokens, "token เข้า ·", resp.usage.completion_tokens, "token ออก")

## 2. ดู token ด้วยตาตัวเอง

`tiktoken` เป็น tokenizer ของ OpenAI (`o200k_base`) ที่รันบนเครื่องเราได้เลย — **ไม่ใช่ของ Gemma**
แต่ช่วยให้เห็นว่าข้อความถูกตัดเป็นชิ้นอย่างไร

In [ ]:
enc = tiktoken.get_encoding("o200k_base")

def show_tokens(text):
    ids = enc.encode(text)
    # ✏️ TODO 1 (▶ วิดีโอ 5:56): แปลงเลข token แต่ละตัวใน ids กลับเป็นข้อความ (ทีละตัว)
    pieces = [___ for i in ids]
    print(len(ids), "token:", pieces)

In [ ]:
show_tokens("unhappiness")
show_tokens("Agents use tools to finish tasks.")
show_tokens("เอเจนต์ใช้เครื่องมือเพื่อทำงานให้เสร็จ")

## 3. นับ token จริงของ Gemma

Gemma มี tokenizer ของตัวเอง วิธีที่แน่นอนที่สุดคือถาม API ให้นับให้ — ส่งข้อความไปพร้อม `max_tokens=1` แล้วอ่าน `usage.prompt_tokens`

In [ ]:
def prompt_tokens(text):
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": text}],
        max_tokens=1, extra_body=EXTRA,
    )
    # ✏️ TODO 2 (▶ วิดีโอ 7:26): คืนจำนวน token ที่ส่งเข้าไป จาก usage ของ resp
    return ___

In [ ]:
# ✏️ TODO 3 (▶ วิดีโอ 7:33): จำนวน token ของ chat template = token ทั้งหมดของ "a" ลบ 1
OVERHEAD = ___
print("chat template เพิ่ม", OVERHEAD, "token")

In [ ]:
for text in ["Agents use tools to finish tasks.", "เอเจนต์ใช้เครื่องมือเพื่อทำงานให้เสร็จ"]:
    gemma = prompt_tokens(text) - OVERHEAD
    print(f"tiktoken {len(enc.encode(text)):>2} · Gemma {gemma:>2} · {text}")

## 4. ค่าใช้จ่ายและ context window

OpenRouter บอกราคาต่อ token และขนาด context window ของทุก model ที่ `https://openrouter.ai/api/v1/models`

In [ ]:
models = requests.get("https://openrouter.ai/api/v1/models").json()["data"]
info = next(m for m in models if m["id"] == MODEL)
PRICE_IN = float(info["pricing"]["prompt"])
PRICE_OUT = float(info["pricing"]["completion"])
CONTEXT = info["context_length"]
print(f"{PRICE_IN * 1e6:.4g} $/ล้าน token เข้า · {PRICE_OUT * 1e6:.4g} $/ล้าน token ออก · context {CONTEXT:,} token")

In [ ]:
# ✏️ TODO 4 (▶ วิดีโอ 9:33): ค่าใช้จ่าย = token เข้า × ราคาเข้า + token ออก × ราคาออก
u = resp.usage
cost = ___
print(f"คำถามแรก ≈ ${cost:.8f} · ถาม 1,000 ครั้ง ≈ ${cost * 1000:.4f}")

In [ ]:
ABSTRACT = """งานวิจัยนี้ศึกษาการใช้ transformer พยากรณ์อนุกรมเวลา เราเสนอวิธีแบ่งข้อมูลเป็นช่วงสั้นๆ
แล้วให้ attention เรียนรู้ความสัมพันธ์ระหว่างช่วง ผลการทดลองกับข้อมูลไฟฟ้าและการจราจร
แสดงว่าความคลาดเคลื่อนลดลงเมื่อเทียบกับ model เดิม และใช้เวลาฝึกน้อยลง"""
n = prompt_tokens(ABSTRACT)
print(f"abstract = {n} token · ใช้ {n / CONTEXT:.3%} ของ context · ใส่ได้ราว {CONTEXT // n:,} ฉบับ")

## 5. คำตอบที่ถูกตัด: `max_tokens` และ `finish_reason`

In [ ]:
for limit in [20, 300]:
    r = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": "อธิบายว่า LLM คืออะไร ในสองประโยค"}],
        max_tokens=limit, extra_body=EXTRA,
    )
    reason = r.choices[0].finish_reason
    print(f"max_tokens={limit} → finish_reason={reason!r} · {r.usage.completion_tokens} token")
    # ✏️ TODO 5 (▶ วิดีโอ 11:25): เช็กว่าคำตอบถูกตัดเพราะชนเพดาน max_tokens หรือไม่
    if ___:
        print("  ✂️ ถูกตัด:", r.choices[0].message.content)

## สรุป

| เราได้เห็น | ตัวเลขจริงจาก lab นี้ |
|---|---|
| chat template เพิ่ม token ทุกครั้งที่เรียก | 13 token |
| ภาษาไทยใช้ token มากกว่าภาษาอังกฤษ | อังกฤษ 7 · ไทย 11 (Gemma) / 15 (tiktoken) |
| model คนละตัว นับ token ไม่เท่ากัน | Gemma 11 vs tiktoken 15 สำหรับประโยคเดียวกัน |
| token ออกแพงกว่า token เข้า | ราว 3 เท่า |
| `finish_reason="length"` = คำตอบถูกตัด | ที่ `max_tokens=20` |

**แบบฝึกหัด:** เขียนคำถามภาษาไทยกับภาษาอังกฤษที่ความหมายเดียวกัน 3 คู่ แล้ววัดว่าภาษาไทยใช้ token มากกว่ากี่เท่าโดยเฉลี่ย (ใช้ `prompt_tokens` และ `OVERHEAD`)

▶ **ถัดไป: Lab 2 — Decoding** เปิดดูความน่าจะเป็นจริงของ Gemma แล้วเขียน temperature, top-k, top-p, min-p เอง